<a href="https://colab.research.google.com/github/ARCHITTOMAR15/Sarcasm-Detection-with-Traditional-NLP-Deep-Learning-and-Transformers/blob/main/SARCSAM_DETECTION_USING_TRANSFORMER_MODELS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
import os
os.listdir('/content/drive/MyDrive/sarcsam_detection')

['Sarcasm_Headlines_Dataset_v2.json']

In [41]:
# import all necessary libraries
import re
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


In [42]:
#data reading
df=pd.read_json('/content/drive/MyDrive/sarcsam_detection/Sarcasm_Headlines_Dataset_v2.json',lines=True)

In [43]:
df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28619 entries, 0 to 28618
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   is_sarcastic  28619 non-null  int64 
 1   headline      28619 non-null  object
 2   article_link  28619 non-null  object
dtypes: int64(1), object(2)
memory usage: 670.9+ KB


In [45]:
df.isnull().sum()

,0
is_sarcastic,0
headline,0
article_link,0


In [46]:
df.duplicated().sum()

np.int64(2)

In [47]:
df["headline"].duplicated().sum()

np.int64(116)

In [48]:
df=df.drop_duplicates(subset=["headline"]).reset_index(drop=True)

In [49]:
df["headline"].duplicated().sum()

np.int64(0)

In [50]:
# Basic minimal clearning
def clean_text(text):
  text=str(text)
  # remove URLs
  text = re.sub(r"http\S+|www\S+", "", text)
  # remove extra spaces
  text = re.sub(r"\s+", " ", text)

  return text.strip()

In [51]:
df["clean_headline"]=df["headline"].apply(clean_text)

In [52]:
df["is_sarcastic"].value_counts()

,count
is_sarcastic,
0,14951
1,13552


In [53]:
df["is_sarcastic"].value_counts(normalize=True)

,proportion
is_sarcastic,
0,0.524541
1,0.475459


In [54]:
#       Create train,test and validation slpit

In [55]:
from sklearn.model_selection import train_test_split

In [56]:
train_df,temp_df=train_test_split(df,test_size=0.20,random_state=42,stratify=df["is_sarcastic"])

In [57]:
# split temp_df into test and validation
val_df,test_df=train_test_split(temp_df,test_size=0.50,random_state=42,stratify=temp_df["is_sarcastic"])

In [58]:
#reset index
train_df=train_df.reset_index(drop=True)
test_df=test_df.reset_index(drop=True)
val_df=val_df.reset_index(drop=True)

In [59]:
print("Train_shape",train_df.shape)
print("Test_shape",test_df.shape)
print("val_shape",val_df.shape)

Train_shape (22802, 4)
Test_shape (2851, 4)
val_shape (2850, 4)


In [60]:
# convert pandas dataframe to hugging face dataset
from datasets import Dataset

In [61]:
train_dataset=Dataset.from_pandas(train_df[["clean_headline","is_sarcastic"]])
test_dataset=Dataset.from_pandas(test_df[["clean_headline","is_sarcastic"]])
val_dataset=Dataset.from_pandas(val_df[["clean_headline","is_sarcastic"]])

In [62]:
# rename is_sarcastc to label
train_dataset=train_dataset.rename_column("is_sarcastic","label")
test_dataset=test_dataset.rename_column("is_sarcastic","label")
val_dataset=val_dataset.rename_column("is_sarcastic","label")